In [ ]:
install.packages("zoo", quiet = TRUE)

In [ ]:
library(zoo, warn.conflicts = FALSE)
library(dplyr, warn.conflicts = FALSE)
library(tidyr)
library(data.table, warn.conflicts = FALSE)
library(arrow, warn.conflicts = FALSE)
library(RPostgres);

### Chargement des données

In [ ]:
# Parameters
ANNEE_A_ANALYSER = 2026
UPLOAD = TRUE

In [ ]:
# paths
PROJECT_BASE_PATH <- '/home/hexa/workspace/pnlp-tdb-pipeline/'
PROJECT_DATA_PATH = paste0(PROJECT_BASE_PATH, 'data/')
OUTPUT_DATA_PATH <- paste0(PROJECT_DATA_PATH, "processed/")

In [ ]:
df <- read_feather(paste0(PROJECT_DATA_PATH, 
                          "interim/PNLP_logistics_", 
                          ANNEE_A_ANALYSER,
                          ".feather"))

setDT(df)

In [ ]:
# need this otherwise way too large (CMM included)
df <- df[COC_NAME %in% c('Stock Disponible Utilisable-SDU', 'MSD',
                         'Nbre de jours de rupture de stock', 
                         'Quantité consommée', 'CMM')]


In [ ]:
sort(unique(df$PERIOD))

In [ ]:
unique(df$COC_NAME)

### Chargement des fonctions

In [ ]:
source(paste0(PROJECT_BASE_PATH, "helpers_logistics.R"))

### Préparation des données

In [ ]:
df_mensuel <- format_time_period(df)

In [ ]:
message("Data ready")

### Indicateurs logistiques

Pour les indicateurs logistiques, nous allons utiliser les Data Elements des différents médicaments/produits et pour chacun d'eux, nous aurrons besoins de 3 COC : 

- Nombre de jours de rupture de stocks : `XEN3ucCGa07`
- Stock Disponible en fin de période : `kOWsLrtvrhn`
- Mois de stocks disponible : `jpmZl19SSta`

Les deux premiers COC sont liées aux Data Element de type Zero or Positive Int alors que la dernière est liées au DE : Number
Pour chaque produit/médicament, il y aura dès lors deux DE à charger, Zero or Positive Int et Number.

La liste des 13 produites retenus est présentées ici : 

Nom des produits suivis

- Artesunate + Amodiaquine, 25mg+67.5mg, Tab, 2-11 mois	`jlRZE5tTARe`, `oDOcYJ4wLRP`

- Artesunate + Amodiaquine, 50mg+135mg, Tab, 1-5 ans	`bdy45DCTJLB`, `ypN69muDHoO`

- Artesunate + Amodiaquine, 100mg+270mg, Tab, 6-13 ans	`Xt48UQAZppa`, `ZJnypGdwZ0G`

- Artesunate + Amodiaquine, 100mg+270mg, Tab, Adulte	`ccpDQ5umc0a`, `JD1dcTpKSic`

- Artesunate, 100 mg, suppositoire, Unité	`FVJ2v5RgBgL`, `eounDZJHMgn`

- Artésunate, 60mg, Injectable, Vial	`b0f8cOW1CFV`, `ZpmFCs2dyxD`

- Sulfadoxine + Pyrimethamine, 500+25mg, Tab, 1000, Vrac	`eKHjiGzfBep`, `UHAfqWdf7WO`

- Test Diagnostic Rapide, Malaria, 25 Tests, Unité	`vtqAD1gqMxx`, `YQG37Uw70oU`

- Arthemether+Luméfantrine, 20 mg+120mg, 6 Tab dispersible, 2-11 mois	`pScfYi2Qxop`, `ONRbtRbf64B`

- Arthemether+Luméfantrine, 20 mg+120mg, 12 Tab dispersible, 1-5 ans	`imqSmhfQTPN`, `bwUSX4FMAEM`

- Arthemether+Luméfantrine, 20 mg+120mg, 18 Tab, 6-13 ans	`hgjnpRWXunB`, `NquDFFwuMRu`

- Arthemether+Luméfantrine, 20 mg+120mg, 24 Tab, Adulte	 `fHyOLs6mPEr`, `HMnLiYTxJBf`

- Moustiquaire Imprégnée d’insecticide (MILD)	`PLjvTAvttMR`, `xqw3Jvrkk1M`


### 4.14.1. % des FOSA n'ayant pas eu de rupture de stocks

Nombre de FOSA par aire de santé/zone de santé / etc qui n'ont connus aucun jour de rupture de stock (Nbr de jour de rupture =0) / Nbr total de FOSA par aire de santé / zone de santé / etc

In [ ]:
L1_nom <- 'Pct_FOSA_Sans_Rupture'

In [ ]:
L1 <- pct_sans_rupture(df_mensuel,
                       indicateur_name=L1_nom)

### 4.14.2. Stock Disponible en fin de période

Cela correspond simplement au SDU (quantité)

In [ ]:
L2_nom <- 'Stock_Disp'

In [ ]:
L2 <- calculer_indicateur(df_mensuel,
                          indicateur_name=L2_nom)

### 4.14.3. Mois de stock disponible

Celas correspond simplement au nombre de mois disponibles

In [ ]:
L3_nom <- 'Mois_stock'

In [ ]:
L3 <- calculer_indicateur(df_mensuel,
                          indicateur_name=L3_nom)

### 4.14.4. Quantités consommées en produits ACT + Artesunate injectable

In [ ]:
L4_nom <- 'ACT_consommé'

In [ ]:
L4 <- calculer_indicateur(df_mensuel,
                          indicateur_name=L4_nom)

### 4.14.5. % des FOSA n'ayant pas eu de rupture de stocks ACT + Artesunate injectable

In [ ]:
L5_nom <- 'Pct_FOSA_Sans_Rupture_ACT'

In [ ]:
L5 <- pct_sans_rupture(df_mensuel,
                       indicateur_name=L5_nom)

### 4.14.6. % des FOSA n'ayant pas eu de rupture de stocks de tous les produits traceurs

In [ ]:
L6_nom <- 'Pct_FOSA_Sans_Rupture_Tous_Produits'

In [ ]:
L6 <- pct_sans_rupture(df_mensuel,
                       indicateur_name=L6_nom)

### 4.15.1. CMM - Cela correspond simplement au quantité

In [ ]:
L7_nom <- 'CMM'  # Look at the selection

In [ ]:
L7 <- calculer_indicateur(df_mensuel,
                          indicateur_name=L7_nom)

### Outputs

In [ ]:
output_final <- do.call(rbind, 
                        list(L1, L2, L3, L4, L5, L6, L7))

In [ ]:
message("File concatenated")
flush.console()

In [ ]:
# s3write_using(x = output_final,
#               FUN = fwrite,
#               object = paste0(DATA_PATH,
#                               "output/PNLP_dash_logistics_",
#                               ANNEE_A_ANALYSER,
#                               "_ND.csv"),
#               opts = list(region = "eu-central-1"))

write_feather(output_final, paste0(OUTPUT_DATA_PATH, 
                    "PNLP_dash_logistics_", 
                    ANNEE_A_ANALYSER,
                    ".feather"))

In [ ]:
message("File saved!")
flush.console()

In [ ]:
# combiner plusieurs années de données

# if(UPLOAD) {

#     #####
#     # test to see if clearing memory of largest 
#     # objects allows process to run
#     rm(df_mensuel, df, output_final)
#     gc()
#     #####
    
#     file.names <- list.files(OUTPUT_DATA_PATH, pattern = 'logistics_202.+feather')
#     file.paths <- paste0(OUTPUT_DATA_PATH, file.names)
    
#     output.data <- lapply(file.paths, read_feather)
#     output.data <- lapply(output.data, setDT)
    
#     output.combined <- rbindlist(output.data)

#     avail_name_dict <- fread(paste0(PROJECT_DATA_PATH, "metadata/",
#                                     "logistics_avail_dict.csv"))
    
    
#     output.combined <- merge(output.combined, avail_name_dict, 
#                              by = 'DE_NAME', all.x = TRUE)
    
#     output.combined <- output.combined[Agg_period < as.yearmon(Sys.Date())]
# }

#### Write to DB

In [ ]:
# if(UPLOAD) {
     
#     con <- dbConnect(
#         RPostgres::Postgres(),
#         dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
#         host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
#         port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
#         user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
#         password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
#     )

#     dbWriteTable(con, "logistics_ND", output.combined, overwrite=TRUE)
# }

In [ ]:
# if (UPLOAD) {
  
#   # Connect to Postgres
#   con <- dbConnect(
#     RPostgres::Postgres(),
#     dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
#     host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
#     port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
#     user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
#     password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
#   )
  
#   # Define chunk size
#   chunk_size <- 50000
#   total_rows <- nrow(output.combined)
#   num_chunks <- ceiling(total_rows / chunk_size)
  
#   # If table exists, overwrite with first chunk
#   for (i in 1:num_chunks) {
#     start_row <- (i - 1) * chunk_size + 1
#     end_row <- min(i * chunk_size, total_rows)
    
#     chunk <- output.combined[start_row:end_row, ]
    
#     dbWriteTable(
#       con,
#       name = "logistics_ND",
#       value = chunk,
#       append = ifelse(i == 1, FALSE, TRUE), # overwrite first chunk, append the rest
#       row.names = FALSE
#     )
#   }
  
#   # Disconnect
#   dbDisconnect(con)
# }

#### Write to DB per file

If the logistics file has already been created, we can just run this cell to push the data per year.

In [ ]:
tail(output_final, 3)

In [ ]:
if (UPLOAD) {
    
    con <- dbConnect(
        RPostgres::Postgres(),
        dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
        host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
        port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
        user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
        password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
    )

    file_path <- file.path(OUTPUT_DATA_PATH, glue::glue('PNLP_dash_logistics_{ANNEE_A_ANALYSER}.feather')) # 1 file CURRENT YEAR
    if (!file.exists(file_path)) stop("Feather file not found: ", file_path)
    
    avail_name_dict <- fread(file.path(PROJECT_DATA_PATH, "metadata/logistics_avail_dict.csv"))
    chunk_size <- 500000

    message("Processing: ", file_path)
    dt <- read_feather(file_path)
    setDT(dt)
    
    dt <- merge(dt, avail_name_dict, by = "DE_NAME", all.x = TRUE)
    dt <- dt[Agg_period < as.yearmon(Sys.Date())]

    # Delete the current YEAR data from the table to replace
    tryCatch({
        if (dbExistsTable(con, "logistics_ND")) {
            dbExecute(
                con,
                sprintf(
                    # "DELETE FROM public.\"logistics_ND\" WHERE substring(\"Agg_period\" from '[0-9]{4}$')::int IN (%s);",
                    # 'DELETE FROM public."logistics_ND" WHERE "Agg_period" LIKE \'%%%s\'',                    
                    # as.character(ANNEE_A_ANALYSER)
                    'DELETE FROM public."logistics_ND" WHERE "Annee" = %d',
                    ANNEE_A_ANALYSER
                )
            )
            message("Data for year deleted: ", ANNEE_A_ANALYSER)
        }
        }, error = function(e) {
                        message("Error deleting data for year ", ANNEE_A_ANALYSER, ": ", e$message)
                        stop(e$message)
            }
    )

    total_rows <- nrow(dt)
    num_chunks <- ceiling(total_rows / chunk_size)
    
    for (i in seq_len(num_chunks)) {
        start_row <- (i - 1) * chunk_size + 1
        end_row <- min(i * chunk_size, total_rows)    
        chunk <- dt[start_row:end_row]
        
        dbWriteTable(
            con,
            name = "logistics_ND",
            value = chunk,
            append = TRUE,
            row.names = FALSE
        )
        message("  Uploaded chunk ", i, "/", num_chunks)
        flush.console()
    }
    rm(dt)
    gc()
    dbDisconnect(con)
}

**The following code was used to revert the DB table to december 2024**

In [ ]:
# options(arrow.unsafe_metadata = FALSE)

# file.names <- c("PNLP_dash_logistics_2020.feather", "PNLP_dash_logistics_2021.feather",
#                 "PNLP_dash_logistics_2022.feather", "PNLP_dash_logistics_2023.feather",
#                 "PNLP_dash_logistics_2024.feather")

# file.paths <- paste0(OUTPUT_DATA_PATH, file.names)

# output.data <- lapply(file.paths, read_feather)
# output.data <- lapply(output.data, setDT)
# output.combined <- rbindlist(output.data)

# # Rename
# avail_name_dict <- fread(paste0(PROJECT_DATA_PATH, "metadata/", "logistics_avail_dict.csv"))
# output.combined <- merge(output.combined, avail_name_dict, by = 'DE_NAME', all.x = TRUE)

# output.combined <- output.combined[Agg_period < as.yearmon(Sys.Date())] # Date filter could be applied here..

In [ ]:
# con <- dbConnect(
#     RPostgres::Postgres(),
#     dbname = Sys.getenv("WORKSPACE_DATABASE_DB_NAME"),
#     host = Sys.getenv("WORKSPACE_DATABASE_HOST"),
#     port = Sys.getenv("WORKSPACE_DATABASE_PORT"),
#     user = Sys.getenv("WORKSPACE_DATABASE_USERNAME"),
#     password = Sys.getenv("WORKSPACE_DATABASE_PASSWORD")
# )

# dbWriteTable(con, "logistics_ND", output.combined, overwrite=TRUE)